# IBM AML Iceberg Queries

This notebook is dedicated to Iceberg/Trino comparisons.
Run first:
`./scripts/iceberg_local_up.sh`
`./scripts/iceberg_seed_trino.sh`


## Optional: Run Local Iceberg Setup From Notebook

This cell can run the local setup scripts directly:
1. `bash ./scripts/iceberg_local_down.sh`
2. `bash ./scripts/iceberg_local_up.sh`
3. `bash ./scripts/iceberg_seed_trino.sh`

`RUN_DOWN` defaults to `False` to avoid deleting volumes unless you opt in.


In [1]:
from pathlib import Path
import subprocess
from IPython.display import display, Markdown

REPO_ROOT = Path.home() / "SourceCode/graph-query-engine"
RUN_DOWN = False
RUN_UP = True
RUN_SEED = True
AUTO_RUN_SETUP = False


def run_local_iceberg_setup(run_down: bool = False, run_up: bool = True, run_seed: bool = True) -> bool:
    if not REPO_ROOT.exists():
        display(Markdown(f"**Error:** repo path not found: `{REPO_ROOT}`"))
        return False

    steps = []
    if run_down:
        steps.append("bash ./scripts/iceberg_local_down.sh")
    if run_up:
        steps.append("bash ./scripts/iceberg_local_up.sh")
    if run_seed:
        steps.append("bash ./scripts/iceberg_seed_trino.sh")

    if not steps:
        display(Markdown("No setup steps selected. Set one of `RUN_DOWN/RUN_UP/RUN_SEED` to `True`."))
        return False

    display(Markdown(f"Running setup in `{REPO_ROOT}`"))
    ok = True
    for i, cmd in enumerate(steps, 1):
        display(Markdown(f"**Step {i}:** `{cmd}`"))
        proc = subprocess.run(cmd, cwd=REPO_ROOT, shell=True, text=True, capture_output=True)
        out = (proc.stdout or "").strip()
        err = (proc.stderr or "").strip()
        if proc.returncode == 0:
            display(Markdown("- Status: OK"))
        else:
            display(Markdown(f"- Status: FAILED (exit {proc.returncode})"))
            ok = False
        if out:
            print(out)
        if err:
            print(err)
        if proc.returncode != 0:
            break

    display(Markdown("**Setup completed successfully.**" if ok else "**Setup failed.**"))
    return ok


if AUTO_RUN_SETUP:
    run_local_iceberg_setup(run_down=RUN_DOWN, run_up=RUN_UP, run_seed=RUN_SEED)
else:
    print("Set AUTO_RUN_SETUP=True and run this cell, or call run_local_iceberg_setup(...)")


Set AUTO_RUN_SETUP=True and run this cell, or call run_local_iceberg_setup(...)


In [2]:
import requests
import pandas as pd
import subprocess
import json as _json
from typing import Dict, Any
from IPython.display import display, Markdown
from pathlib import Path

BASE_URL = "http://localhost:7000"
ICEBERG_MAPPING_PATH = str(Path.cwd() / "mappings/iceberg-local-mapping.json")
ICEBERG_MAPPING_ID = "iceberg-local"
TRINO_CONTAINER = "iceberg-trino"
TRINO_SERVER = "http://localhost:8080"
TRINO_CATALOG = "iceberg"
TRINO_SCHEMA = "aml"


In [3]:
def get_sql_explain(gremlin_query: str) -> Dict[str, Any]:
    try:
        response = requests.post(
            f"{BASE_URL}/query/explain",
            json={"gremlin": gremlin_query},
            headers={"Content-Type": "application/json"},
            timeout=10,
        )
        if response.ok:
            return response.json()
        return {"error": f"HTTP {response.status_code}: {response.text}"}
    except Exception as e:
        return {"error": str(e)}


def upload_iceberg_mapping() -> bool:
    try:
        with open(ICEBERG_MAPPING_PATH, "rb") as f:
            resp = requests.post(
                f"{BASE_URL}/mapping/upload",
                params={"id": ICEBERG_MAPPING_ID, "name": "Iceberg Local", "activate": "false"},
                files={"file": ("iceberg-local-mapping.json", f, "application/json")},
                timeout=15,
            )
        m = resp.json()
        if "error" not in m:
            display(Markdown(f"Iceberg mapping ready - id=`{m.get('mappingId')}`"))
            return True
        display(Markdown(f"**Mapping upload failed:** {m['error']}"))
        return False
    except Exception as e:
        display(Markdown(f"**Mapping upload error:** {e}"))
        return False


def get_iceberg_sql(gremlin_query: str) -> tuple[str, list, str | None]:
    try:
        resp = requests.post(
            f"{BASE_URL}/query/explain",
            json={"gremlin": gremlin_query},
            headers={"Content-Type": "application/json", "X-Mapping-Id": ICEBERG_MAPPING_ID},
            timeout=10,
        )
        if resp.ok:
            data = resp.json()
            return data.get("translatedSql", ""), data.get("parameters", []), None
        err = None
        try:
            err = resp.json().get("error")
        except Exception:
            err = resp.text
        return "", [], f"HTTP {resp.status_code}: {err}"
    except Exception as e:
        return "", [], str(e)


def _sql_literal(value) -> str:
    if value is None:
        return "NULL"
    if isinstance(value, bool):
        return "TRUE" if value else "FALSE"
    if isinstance(value, (int, float)):
        return str(value)
    s = str(value).replace("'", "''")
    return f"'{s}'"


def bind_sql_parameters(sql: str, params: list) -> str:
    out = sql
    for p in params:
        if "?" not in out:
            raise ValueError("More SQL parameters were supplied than placeholders")
        out = out.replace("?", _sql_literal(p), 1)
    if "?" in out:
        raise ValueError("SQL still contains unbound placeholders")
    return out


def run_trino(sql: str) -> pd.DataFrame:
    result = subprocess.run(
        ["docker", "exec", "-i", TRINO_CONTAINER, "trino", "--server", TRINO_SERVER, "--catalog", TRINO_CATALOG, "--schema", TRINO_SCHEMA, "--output-format", "JSON", "--execute", sql],
        capture_output=True,
        text=True,
        timeout=30,
    )
    if result.returncode != 0:
        stderr_lines = [ln.strip() for ln in (result.stderr or "").splitlines() if ln.strip()]
        useful = [ln for ln in stderr_lines if "org.jline.utils.Log" not in ln]
        msg = (useful[-1] if useful else (stderr_lines[-1] if stderr_lines else "Trino execution failed"))
        return pd.DataFrame([{"error": msg[:500]}])
    rows = [_json.loads(l) for l in result.stdout.strip().splitlines() if l.strip()]
    return pd.DataFrame(rows if rows else [{"result": "empty"}])


def compare_sql_and_iceberg(title: str, gremlin: str):
    display(Markdown(f"### {title}"))
    display(Markdown(f"**Gremlin:** `{gremlin}`"))
    ice_sql, ice_params, ice_err = get_iceberg_sql(gremlin)
    if not ice_sql:
        if ice_err:
            display(Markdown(f"*Iceberg SQL translation not available: {ice_err}*"))
        else:
            display(Markdown("*Iceberg SQL translation not available for this query.*"))
        return
    display(Markdown(f"```sql\n{ice_sql}\n```"))
    if ice_params:
        display(Markdown(f"**Parameters:** `{ice_params}`"))
    try:
        executable_sql = bind_sql_parameters(ice_sql, ice_params)
    except Exception as e:
        display(Markdown(f"*Failed to bind SQL parameters: {e}*"))
        return
    display(run_trino(executable_sql))


upload_iceberg_mapping()


Iceberg mapping ready - id=`iceberg-local`

True

## Quick Verify

Run this cell to verify backend health, Trino connectivity, and seeded Iceberg tables.


In [4]:
def quick_verify_iceberg() -> bool:
    checks = []

    try:
        health_resp = requests.get(f"{BASE_URL}/health", timeout=5)
        health_ok = False
        health_detail = ""

        if health_resp.ok:
            try:
                payload = health_resp.json()
                if isinstance(payload, dict):
                    health_ok = str(payload.get("status", "")).strip().upper() == "OK"
                    health_detail = _json.dumps(payload)
                else:
                    text_payload = str(payload).strip()
                    health_ok = text_payload.upper() == "OK"
                    health_detail = text_payload
            except ValueError:
                text_payload = (health_resp.text or "").strip()
                health_ok = text_payload.upper() == "OK"
                health_detail = text_payload
        else:
            health_detail = f"HTTP {health_resp.status_code}: {(health_resp.text or '').strip()}"

        checks.append(("Backend", health_ok, health_detail))
    except Exception as e:
        checks.append(("Backend", False, str(e)))

    trino_ping = run_trino("SELECT 1 AS ok")
    trino_ok = "error" not in trino_ping.columns
    trino_detail = "ok" if trino_ok else trino_ping.to_string(index=False)[:240]
    checks.append(("Trino", trino_ok, trino_detail))

    table_probe = run_trino("SELECT count(*) AS c FROM accounts")
    table_ok = "error" not in table_probe.columns
    table_count = None
    if table_ok and "c" in table_probe.columns and len(table_probe.index) > 0:
        table_count = table_probe.iloc[0]["c"]
    table_detail = f"accounts={table_count}" if table_ok else table_probe.to_string(index=False)[:240]
    checks.append(("Seed data", table_ok, table_detail))

    all_ok = all(ok for _, ok, _ in checks)
    summary = "PASS" if all_ok else "FAIL"
    display(Markdown(f"**Health summary:** `{summary}`"))

    for name, ok, detail in checks:
        status = "PASS" if ok else "FAIL"
        display(Markdown(f"- **{name}:** `{status}`"))
        if not ok and detail:
            display(Markdown(f"  - detail: `{detail}`"))

    if not all_ok:
        display(Markdown("**Action:** run `run_local_iceberg_setup(run_down=False, run_up=True, run_seed=True)` then retry."))
    return all_ok


quick_verify_iceberg()


**Health summary:** `PASS`

- **Backend:** `PASS`

- **Trino:** `PASS`

- **Seed data:** `PASS`

True

## Query Sections

This notebook mirrors all query titles from `aml_demo_queries.ipynb` and runs each via Iceberg translation + Trino execution.


## Simple Queries


### S1 Count accounts

Count unique Account vertices.


In [5]:
compare_sql_and_iceberg(
    title='S1 Count accounts',
    gremlin="g.V().hasLabel('Account').count()",
)


### S1 Count accounts

**Gremlin:** `g.V().hasLabel('Account').count()`

```sql
SELECT COUNT(*) AS count FROM aml.accounts
```

,count
0,81352


### S2 Count banks

Count Bank vertices - one per distinct bank ID in the data.


In [6]:
compare_sql_and_iceberg(
    title='S2 Count banks',
    gremlin="g.V().hasLabel('Bank').count()",
)


### S2 Count banks

**Gremlin:** `g.V().hasLabel('Bank').count()`

```sql
SELECT COUNT(*) AS count FROM aml.banks
```

,count
0,4598


### S3 Count countries

Count Country vertices (10 pre-seeded jurisdictions).


In [7]:
compare_sql_and_iceberg(
    title='S3 Count countries',
    gremlin="g.V().hasLabel('Country').count()",
)


### S3 Count countries

**Gremlin:** `g.V().hasLabel('Country').count()`

```sql
SELECT COUNT(*) AS count FROM aml.countries
```

,count
0,10


### S4 Count alerts

Count Alert vertices - one raised per suspicious transfer.


In [8]:
compare_sql_and_iceberg(
    title='S4 Count alerts',
    gremlin="g.V().hasLabel('Alert').count()",
)


### S4 Count alerts

**Gremlin:** `g.V().hasLabel('Alert').count()`

```sql
SELECT COUNT(*) AS count FROM aml.alerts
```

,count
0,5


### S5 Count transfers

Count all TRANSFER edges.


In [9]:
compare_sql_and_iceberg(
    title='S5 Count transfers',
    gremlin="g.E().hasLabel('TRANSFER').count()",
)


### S5 Count transfers

**Gremlin:** `g.E().hasLabel('TRANSFER').count()`

```sql
SELECT COUNT(*) AS count FROM aml.transfers
```

,count
0,100000


### S6 Suspicious transfer count

Count confirmed suspicious TRANSFER edges (isLaundering=1).


In [10]:
compare_sql_and_iceberg(
    title='S6 Suspicious transfer count',
    gremlin="g.E().has('isLaundering','1').count()",
)


### S6 Suspicious transfer count

**Gremlin:** `g.E().has('isLaundering','1').count()`

```sql
SELECT COUNT(*) AS count FROM aml.transfers WHERE is_laundering = ?
```

**Parameters:** `['1']`

,count
0,5


### S7 High-risk countries

List Country vertices with riskLevel=HIGH.


In [11]:
compare_sql_and_iceberg(
    title='S7 High-risk countries',
    gremlin="g.V().hasLabel('Country').has('riskLevel','HIGH').project('countryCode','countryName','region','fatfBlacklist').by('countryCode').by('countryName').by('region').by('fatfBlacklist')",
)


### S7 High-risk countries

**Gremlin:** `g.V().hasLabel('Country').has('riskLevel','HIGH').project('countryCode','countryName','region','fatfBlacklist').by('countryCode').by('countryName').by('region').by('fatfBlacklist')`

```sql
SELECT v.country_code AS "countryCode", v.country_name AS "countryName", v.region AS "region", v.fatf_blacklist AS "fatfBlacklist" FROM aml.countries v WHERE v.risk_level = ?
```

**Parameters:** `['HIGH']`

,countryCode,countryName,region,fatfBlacklist
0,NG,Nigeria,Africa,false
1,KY,Cayman Islands,Americas,true
2,PA,Panama,Americas,true


### S8 High-severity alerts

Show HIGH-severity open alerts.


In [12]:
compare_sql_and_iceberg(
    title='S8 High-severity alerts',
    gremlin="g.V().hasLabel('Alert').has('severity','HIGH').project('alertId','alertType','status','raisedAt').by('alertId').by('alertType').by('status').by('raisedAt').limit(15)",
)


### S8 High-severity alerts

**Gremlin:** `g.V().hasLabel('Alert').has('severity','HIGH').project('alertId','alertType','status','raisedAt').by('alertId').by('alertType').by('status').by('raisedAt').limit(15)`

```sql
SELECT v.alert_id AS "alertId", v.alert_type AS "alertType", v.status AS "status", v.raised_at AS "raisedAt" FROM aml.alerts v WHERE v.severity = ? LIMIT 15
```

**Parameters:** `['HIGH']`

,alertId,alertType,status,raisedAt
0,ALERT-4743,SUSPICIOUS_TRANSFER,OPEN,2022/09/01 00:21
1,ALERT-85764,SUSPICIOUS_TRANSFER,OPEN,2022/09/01 00:03


## Complex Queries


### C1 Top sender accounts

Accounts ranked by outgoing transfer count - find the biggest hubs.


In [13]:
compare_sql_and_iceberg(
    title='C1 Top sender accounts',
    gremlin="g.V().hasLabel('Account').project('accountId','bankId','outDegree').by('accountId').by('bankId').by(outE('TRANSFER').count()).order().by(select('outDegree'),Order.desc).limit(15)",
)


### C1 Top sender accounts

**Gremlin:** `g.V().hasLabel('Account').project('accountId','bankId','outDegree').by('accountId').by('bankId').by(outE('TRANSFER').count()).order().by(select('outDegree'),Order.desc).limit(15)`

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT COUNT(*) FROM aml.transfers WHERE out_id = v.id) AS "outDegree" FROM aml.accounts v ORDER BY "outDegree" DESC LIMIT 15
```

,accountId,bankId,outDegree
0,100428660,070,1924
1,8001409E0,001,13
2,80026D340,010,13
3,800631890,01601,13
4,8001C3570,010,12
5,8008F0D20,021174,12
6,8001523C0,010,11
7,80274CA60,00701,11
8,800D38650,021258,11
9,8001F3760,010,11


### C2 Suspicious hubs

Accounts with the most suspicious outgoing transfers.


In [14]:
compare_sql_and_iceberg(
    title='C2 Suspicious hubs',
    gremlin="g.V().hasLabel('Account').project('accountId','bankId','suspiciousOut','totalOut').by('accountId').by('bankId').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).where(select('suspiciousOut').is(gt(0))).order().by(select('suspiciousOut'),Order.desc).limit(15)",
)


### C2 Suspicious hubs

**Gremlin:** `g.V().hasLabel('Account').project('accountId','bankId','suspiciousOut','totalOut').by('accountId').by('bankId').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).where(select('suspiciousOut').is(gt(0))).order().by(select('suspiciousOut'),Order.desc).limit(15)`

```sql
SELECT * FROM (SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT COUNT(*) FROM aml.transfers WHERE out_id = v.id AND is_laundering = '1') AS "suspiciousOut", (SELECT COUNT(*) FROM aml.transfers WHERE out_id = v.id) AS "totalOut" FROM aml.accounts v) p WHERE p."suspiciousOut" > 0 ORDER BY "suspiciousOut" DESC LIMIT 15
```

,accountId,bankId,suspiciousOut,totalOut
0,100428660,070,5,1924


### C3 Account -> Bank (BELONGS_TO)

Show which bank each account belongs to via BELONGS_TO.


In [15]:
compare_sql_and_iceberg(
    title='C3 Account -> Bank (BELONGS_TO)',
    gremlin="g.V().hasLabel('Account').limit(15).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())",
)


### C3 Account -> Bank (BELONGS_TO)

**Gremlin:** `g.V().hasLabel('Account').limit(15).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())`

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT ARRAY_JOIN(ARRAY_AGG(_njv0.bank_name), ',') FROM aml.account_bank _nje0 JOIN aml.banks _njv0 ON _njv0.id = _nje0.in_id WHERE _nje0.out_id = v.id) AS "bankName" FROM aml.accounts v LIMIT 15
```

,accountId,bankId,bankName
0,805C606A0,029251,Bank-029251
1,809320130,004503,Bank-004503
2,80A048920,012719,Bank-012719
3,80A0493B0,0114674,Bank-0114674
4,80C2FD660,014077,Bank-014077
5,80C2FE020,0132306,Bank-0132306
6,804799410,035893,Bank-035893
7,80D008C00,0218472,Bank-0218472
8,8016A3510,011474,Bank-011474
9,80C2FC690,0231004,Bank-0231004


### C4 Bank -> Country (LOCATED_IN)

Show which country each bank is headquartered in.


In [16]:
compare_sql_and_iceberg(
    title='C4 Bank -> Country (LOCATED_IN)',
    gremlin="g.V().hasLabel('Bank').limit(15).project('bankId','bankName','countryCode','countryName').by('bankId').by('bankName').by('countryCode').by(out('LOCATED_IN').values('countryName').fold())",
)


### C4 Bank -> Country (LOCATED_IN)

**Gremlin:** `g.V().hasLabel('Bank').limit(15).project('bankId','bankName','countryCode','countryName').by('bankId').by('bankName').by('countryCode').by(out('LOCATED_IN').values('countryName').fold())`

```sql
SELECT v.bank_id AS "bankId", v.bank_name AS "bankName", v.country_code AS "countryCode", (SELECT ARRAY_JOIN(ARRAY_AGG(_njv0.country_name), ',') FROM aml.bank_country _nje0 JOIN aml.countries _njv0 ON _njv0.id = _nje0.in_id WHERE _nje0.out_id = v.id) AS "countryName" FROM aml.banks v LIMIT 15
```

,bankId,bankName,countryCode,countryName
0,034816,Bank-034816,US,United States
1,0237730,Bank-0237730,KY,Cayman Islands
2,0025847,Bank-0025847,DE,Germany
3,03208,Bank-03208,CH,Switzerland
4,001,Bank-001,SG,Singapore
5,034200,Bank-034200,NG,Nigeria
6,010,Bank-010,SG,Singapore
7,037785,Bank-037785,KY,Cayman Islands
8,0314561,Bank-0314561,AE,UAE
9,0312561,Bank-0312561,HK,Hong Kong


### C5 Two-hop: Account -> Bank -> Country

Traverse Account->Bank->Country in two hops.


In [17]:
compare_sql_and_iceberg(
    title='C5 Two-hop: Account -> Bank -> Country',
    gremlin="g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)",
)


### C5 Two-hop: Account -> Bank -> Country

**Gremlin:** `g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)`

```sql
SELECT v0.account_id AS accountId0, v1.bank_name AS bankName1, v2.country_name AS countryName2 FROM (SELECT * FROM aml.accounts LIMIT 1) v0 JOIN aml.account_bank e1 ON e1.out_id = v0.id JOIN aml.banks v1 ON v1.id = e1.in_id JOIN aml.bank_country e2 ON e2.out_id = v1.id JOIN aml.countries v2 ON v2.id = e2.in_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) LIMIT 10
```

,accountId0,bankName1,countryName2
0,802E6C0E0,Bank-011974,United States


### C6 Accounts sending to high-risk countries (SENT_VIA)

Find accounts routing money via FATF-blacklisted countries.


In [18]:
compare_sql_and_iceberg(
    title='C6 Accounts sending to high-risk countries (SENT_VIA)',
    gremlin="g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(20)",
)


### C6 Accounts sending to high-risk countries (SENT_VIA)

**Gremlin:** `g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(20)`

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", v.risk_score AS "riskScore" FROM aml.accounts v WHERE EXISTS (SELECT 1 FROM aml.account_country _we JOIN aml.countries _wv ON _wv.id = _we.in_id WHERE _we.out_id = v.id AND _wv.fatf_blacklist = ?) LIMIT 20
```

**Parameters:** `['true']`

,accountId,bankId,riskScore
0,805C90FE0,00513,0.1
1,805C97E10,021611,0.1
2,805CA2440,001,0.1
3,805CBC010,011405,0.1
4,805CC22F0,038109,0.1
5,805CC1050,0215266,0.1
6,805CC3160,0013432,0.1
7,805CC5E20,009129,0.1
8,805CC5B40,01674,0.1
9,805CC8E20,017327,0.1


### C7 Flagged accounts with alert detail (FLAGGED_BY)

Show investigated accounts with linked Alert severity.


In [19]:
compare_sql_and_iceberg(
    title='C7 Flagged accounts with alert detail (FLAGGED_BY)',
    gremlin="g.V().hasLabel('Account').where(outE('FLAGGED_BY')).project('accountId','bankId','alertCount','highSeverity').by('accountId').by('bankId').by(outE('FLAGGED_BY').count()).by(out('FLAGGED_BY').has('severity','HIGH').count()).limit(20)",
)


### C7 Flagged accounts with alert detail (FLAGGED_BY)

**Gremlin:** `g.V().hasLabel('Account').where(outE('FLAGGED_BY')).project('accountId','bankId','alertCount','highSeverity').by('accountId').by('bankId').by(outE('FLAGGED_BY').count()).by(out('FLAGGED_BY').has('severity','HIGH').count()).limit(20)`

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT COUNT(*) FROM aml.account_alert WHERE out_id = v.id) AS "alertCount", (SELECT COUNT(*) FROM aml.account_alert _e JOIN aml.alerts _tv ON _tv.id = _e.in_id WHERE _e.out_id = v.id AND _tv.severity = 'HIGH') AS "highSeverity" FROM aml.accounts v WHERE EXISTS (SELECT 1 FROM aml.account_alert we WHERE we.out_id = v.id) LIMIT 20
```

,accountId,bankId,alertCount,highSeverity
0,100428660,070,5,2


### C8 Cross-bank suspicious flow

Suspicious transfers that cross bank boundaries.


In [20]:
compare_sql_and_iceberg(
    title='C8 Cross-bank suspicious flow',
    gremlin="g.E().has('isLaundering','1').project('fromBank','fromAcct','toBank','toAcct','amount','currency').by(outV().values('bankId')).by(outV().values('accountId')).by(inV().values('bankId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)",
)


### C8 Cross-bank suspicious flow

**Gremlin:** `g.E().has('isLaundering','1').project('fromBank','fromAcct','toBank','toAcct','amount','currency').by(outV().values('bankId')).by(outV().values('accountId')).by(inV().values('bankId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)`

```sql
SELECT ov.bank_id AS "fromBank", ov.account_id AS "fromAcct", iv.bank_id AS "toBank", iv.account_id AS "toAcct", e.amount AS "amount", e.currency AS "currency" FROM aml.transfers e JOIN aml.accounts ov ON ov.id = e.out_id JOIN aml.accounts iv ON iv.id = e.in_id WHERE e.is_laundering = ? LIMIT 15
```

**Parameters:** `['1']`

,fromBank,fromAcct,toBank,toAcct,amount,currency
0,070,100428660,011474,805B716C0,29024.33,US Dollar
1,070,100428660,0032375,80E480620,14288.83,US Dollar
2,070,100428660,0113798,80DC756E0,13171425.53,US Dollar
3,070,100428660,001124,800825340,389769.39,US Dollar
4,070,100428660,015980,80B39E7B0,792.92,US Dollar


### C9 Three-hop money trail

Follow suspicious TRANSFER hops 3 steps deep.


In [21]:
compare_sql_and_iceberg(
    title='C9 Three-hop money trail',
    gremlin="g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)",
)


### C9 Three-hop money trail

**Gremlin:** `g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)`

```sql
SELECT v0.account_id AS accountId0, v1.account_id AS accountId1, v2.account_id AS accountId2, v3.account_id AS accountId3 FROM (SELECT * FROM aml.accounts WHERE EXISTS (SELECT 1 FROM aml.transfers we WHERE we.out_id = id AND we.is_laundering = ?) LIMIT 1) v0 JOIN aml.transfers e1 ON e1.out_id = v0.id JOIN aml.accounts v1 ON v1.id = e1.in_id JOIN aml.transfers e2 ON e2.out_id = v1.id JOIN aml.accounts v2 ON v2.id = e2.in_id JOIN aml.transfers e3 ON e3.out_id = v2.id JOIN aml.accounts v3 ON v3.id = e3.in_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) AND v3.id NOT IN (v0.id, v1.id, v2.id) LIMIT 10
```

**Parameters:** `['1']`

,result
0,empty


### C10 Five-hop layering chain

Extended 5-hop traversal for layering detection.


In [22]:
compare_sql_and_iceberg(
    title='C10 Five-hop layering chain',
    gremlin="g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(5).path().by('accountId').limit(10)",
)


### C10 Five-hop layering chain

**Gremlin:** `g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(5).path().by('accountId').limit(10)`

```sql
SELECT v0.account_id AS accountId0, v1.account_id AS accountId1, v2.account_id AS accountId2, v3.account_id AS accountId3, v4.account_id AS accountId4, v5.account_id AS accountId5 FROM (SELECT * FROM aml.accounts WHERE EXISTS (SELECT 1 FROM aml.transfers we WHERE we.out_id = id AND we.is_laundering = ?) LIMIT 1) v0 JOIN aml.transfers e1 ON e1.out_id = v0.id JOIN aml.accounts v1 ON v1.id = e1.in_id JOIN aml.transfers e2 ON e2.out_id = v1.id JOIN aml.accounts v2 ON v2.id = e2.in_id JOIN aml.transfers e3 ON e3.out_id = v2.id JOIN aml.accounts v3 ON v3.id = e3.in_id JOIN aml.transfers e4 ON e4.out_id = v3.id JOIN aml.accounts v4 ON v4.id = e4.in_id JOIN aml.transfers e5 ON e5.out_id = v4.id JOIN aml.accounts v5 ON v5.id = e5.in_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) AND v3.id NOT IN (v0.id, v1.id, v2.id) AND v4.id NOT IN (v0.id, v1.id, v2.id, v3.id) AND v5.id NOT IN (v0.id, v1.id, v2.id, v3.id, v4.id) LIMIT 10
```

**Parameters:** `['1']`

,result
0,empty


### C11 Transactional suspicious count

Suspicious transfer count via transactional endpoint.


In [23]:
compare_sql_and_iceberg(
    title='C11 Transactional suspicious count',
    gremlin="g.E().has('isLaundering','1').count()",
)


### C11 Transactional suspicious count

**Gremlin:** `g.E().has('isLaundering','1').count()`

```sql
SELECT COUNT(*) AS count FROM aml.transfers WHERE is_laundering = ?
```

**Parameters:** `['1']`

,count
0,5


## Done
